<a href="https://colab.research.google.com/github/mk654/SML_PG60/blob/main/COMP90051_ProjectGroup60_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **COMP90051 Group Project → Code**

| Project Group 60 |         |
|------------------|---------|
| Lachlan Fox      | 649622  |
| Songhao Guo      | 1542657 |
| Amelia King      | 1175861 |


github → https://github.com/mk654/SML_PG60


In [ ]:
# RUN FIRST
from scipy.io import loadmat
import pandas as pd
from pathlib import Path
import numpy as np
from scipy import sparse


# Amelia -> load data from github repo
!git clone https://github.com/mk654/SML_PG60
repo_dir = Path("/content/SML_PG60")

flu_mat = repo_dir / "data" / "matraw" / "influenza_outbreak_dataset.mat" # access influenza dataset from gitrepo
fludata = loadmat(flu_mat)



# url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/influenza_outbreak_dataset.mat" # old access -> directory = main
#url = "https://raw.githubusercontent.com/mk654/SML_PG60/main/data/influenza_outbreak_dataset.mat" # old access -> directory = main/data


fatal: destination path 'SML_PG60' already exists and is not an empty directory.


### 0.a) inspecting data

*Amelia*


#### *Results*
| name     | outer dtype | outer shape | inner type | inner shape |
|----------|-------------|-------------|------------|-------------|
| X train  | object      | (1, 48)     | csc_matrix | (1095, 545) |
| X test   | object      | (1, 48)     | csc_matrix | (485, 545)  |
| y train  | object      | (1, 48)     | ndarray    | (1095, 1)   |
| y test   | object      | (1, 48)     | ndarray    | (485, 1)    |
| locs     | object      | (1, 48)     | ndarray    | (1,)        |
| keywords | object      | (1, 525)    | ndarray    | (1,)        |

in sum:
- 48 training feature matrices, one per location
- 48 testing feature matrices, one per location
- 48 training label vectors
- 48 testing label vectors
- 48 location names/IDs
- 545 keyword feature names



*Code*


```
rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)
```

In [ ]:
# delete this cell before submission!!

rows = []
for name in ["flu_X_tr", "flu_X_te", "flu_Y_tr", "flu_Y_te", "flu_locs", "flu_keywords"]:
    value = fludata[name]
    try:
        first = value[0, 0]
        first_type = type(first).__name__
        first_shape = getattr(first, 'shape', None)
    except:
        first_shape = None
        first_type = None
    rows.append({
        "name": name,
        "outer_dtype": value.dtype,
        "outer_shape": value.shape,
        "inner_type": first_type,
        "inner_shape": first_shape
    })
df_summary = pd.DataFrame(rows)
print(df_summary)

           name outer_dtype outer_shape inner_type  inner_shape
0      flu_X_tr      object     (1, 48)  csc_array  (1095, 545)
1      flu_X_te      object     (1, 48)  csc_array   (485, 545)
2      flu_Y_tr      object     (1, 48)    ndarray    (1095, 1)
3      flu_Y_te      object     (1, 48)    ndarray     (485, 1)
4      flu_locs      object     (1, 48)    ndarray         (1,)
5  flu_keywords      object    (1, 525)    ndarray         (1,)


### 0.b) convert .mat files to .csv
Converts influenza_outbreak_dataset.mat to csv

github directory = ```SML_PG60/data/processed/flu_csv```

----
*Note that these csv files are only processed in the sense that they have been converted from .mat to .csv; no further processing has yet taken place.*

### Amelia's conversion
- conversion will produce 48 separate folders aligning with the 48 folds. Each folder will contain:
  1. X_train.csv
  2. X_test.csv
  3. y_train.csv
  4. y_test.csv
  * *note that influenza_outbreak_dataset.mat contains 48 folds (test/train splits). Each fold contains its own X & y train and X & y test. Hence, the data is split and converted as cleanly as possible to avoid errors that may come with combining folds*
- Additionally, the conversion will also produce:
  1. keywords.csv
  2. locs.csv

---

The converted files can be manually downloaded from this notebook by copy/pasting & running the following code:
```
from google.colab import files
!zip -r flu_csv.zip /content/SML_PG60/data/processed/flu_csv
files.download("flu_csv.zip")
```

In [ ]:
# Amelia -> convert influenza_outbreak_dataset.mat to .csv file
flu_out = repo_dir / "data" / "processed" / "flu_csv"
flu_out.mkdir(parents=True, exist_ok=True)

X_tr = fludata["flu_X_tr"]
X_te = fludata["flu_X_te"]
y_tr = fludata["flu_Y_tr"]
y_te = fludata["flu_Y_te"]

n_folds = X_tr.shape[1]

for i in range(n_folds):
    fold_dir = flu_out / f"fold_{i:02d}"
    fold_dir.mkdir(exist_ok=True)

    Xtr = X_tr[0, i].toarray() # some data stored as sparse matrix, convert to dense
    Xte = X_te[0, i].toarray()
    ytr = y_tr[0, i].ravel()
    yte = y_te[0, i].ravel()

    pd.DataFrame(Xtr).to_csv(fold_dir / "X_train.csv", index=False)
    pd.DataFrame(Xte).to_csv(fold_dir / "X_test.csv", index=False)
    pd.DataFrame(ytr).to_csv(fold_dir / "y_train.csv", index=False)
    pd.DataFrame(yte).to_csv(fold_dir / "y_test.csv", index=False)

    print(f"Saved fold {i}")

keywords = fludata["flu_keywords"]
keywords_list = [str(k[0]) for k in keywords.ravel()]
pd.DataFrame(keywords_list, columns=["keyword"]) \
  .to_csv(flu_out / "keywords.csv", index=False)

locs = fludata["flu_locs"]
locs_list = [str(l[0]) for l in locs.ravel()]
pd.DataFrame(locs_list, columns=["location"]) \
  .to_csv(flu_out / "locs.csv", index=False)

Saved fold 0
Saved fold 1
Saved fold 2
Saved fold 3
Saved fold 4
Saved fold 5
Saved fold 6
Saved fold 7
Saved fold 8
Saved fold 9
Saved fold 10
Saved fold 11
Saved fold 12
Saved fold 13
Saved fold 14
Saved fold 15
Saved fold 16
Saved fold 17
Saved fold 18
Saved fold 19
Saved fold 20
Saved fold 21
Saved fold 22
Saved fold 23
Saved fold 24
Saved fold 25
Saved fold 26
Saved fold 27
Saved fold 28
Saved fold 29
Saved fold 30
Saved fold 31
Saved fold 32
Saved fold 33
Saved fold 34
Saved fold 35
Saved fold 36
Saved fold 37
Saved fold 38
Saved fold 39
Saved fold 40
Saved fold 41
Saved fold 42
Saved fold 43
Saved fold 44
Saved fold 45
Saved fold 46
Saved fold 47


### Songhao's conversion
- combines all information from influenza_outbreak_dataset.mat into  ```influenza_outbreak_long.csv``` (>150MB)
- original X matrices within .mat file contain 545 features, however, only 525 named keyword features exist
  * thus ```influenza_outbreak_long.csv``` drops last 20 features within original X matrices
- adds column "time_index" which counts rows in each train/test split per location

In [ ]:
# (Songhao) convert loaded .mat variables to CSV

csv_output_path = repo_dir / "data" / "processed" / "flu_csv" / "influenza_outbreak_long.csv"
keywords_output_path = repo_dir / "data" / "processed" / "flu_csv" / "flu_keywords.csv"
locations_output_path = repo_dir / "data" / "processed" / "flu_csv" / "flu_locations.csv"

def extract_matlab_string_array(arr):
    values = []
    for item in arr.flatten():
        if isinstance(item, np.ndarray):
            if item.size == 1:
                values.append(str(item.item()).strip())
            else:
                values.append("".join(item.astype(str).flatten()).strip())
        else:
            values.append(str(item).strip())
    return values


def make_safe_unique_names(names):
    safe_names = []
    used = {}

    for name in names:
        clean = (
            str(name)
            .strip()
            .replace(" ", "_")
            .replace("-", "_")
            .replace("/", "_")
            .replace("(", "")
            .replace(")", "")
        )

        if clean == "":
            clean = "unnamed_feature"

        if clean in used:
            used[clean] += 1
            clean = f"{clean}_{used[clean]}"
        else:
            used[clean] = 0

        safe_names.append(clean)

    return safe_names

#-------------------------------#
locs = fludata["flu_locs"]
keywrds = fludata["flu_keywords"]
X_tr = fludata["flu_X_tr"]
y_tr = fludata["flu_Y_tr"]
X_te = fludata["flu_X_te"]
y_te = fludata["flu_Y_te"]
#-------------------------------#

locations = extract_matlab_string_array(locs)
keywords = extract_matlab_string_array(keywrds)
feature_names = make_safe_unique_names(keywords)

all_parts = []

for i, location in enumerate(locations):
    for split_name, X_cell, y_cell in [
        ("train", X_tr, y_tr),
        ("test", X_te, y_te),
    ]:
        X = X_cell[0, i]

        if sparse.issparse(X):
            X = X.toarray()
        else:
            X = np.asarray(X)

        # Keep only the 525 named keyword features --> original X matrices contain 545 features (final 20 columns are dropped)
        X = X[:, :len(feature_names)]

        y = np.asarray(y_cell[0, i]).reshape(-1).astype(int)

        df_part = pd.DataFrame(X, columns=feature_names)
        df_part.insert(0, "location", location)
        df_part.insert(1, "split", split_name)
        df_part.insert(2, "time_index", np.arange(len(y)))
        df_part["label"] = y

        all_parts.append(df_part)

df = pd.concat(all_parts, ignore_index=True)

df.to_csv(csv_output_path, index=False)
pd.DataFrame({"keyword": keywords}).to_csv(keywords_output_path, index=False)
pd.DataFrame({"location": locations}).to_csv(locations_output_path, index=False)

print("Saved main CSV to:", csv_output_path)
print("Saved keywords CSV to:", keywords_output_path)
print("Saved locations CSV to:", locations_output_path)

print("\nData shape:", df.shape)
print("\nSplit counts:")
print(df["split"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts())

df.head()

Saved main CSV to: /content/SML_PG60/influenza_outbreak_long.csv
Saved keywords CSV to: /content/SML_PG60/flu_keywords.csv
Saved locations CSV to: /content/SML_PG60/flu_locations.csv

Data shape: (75840, 529)

Split counts:
split
train    52560
test     23280
Name: count, dtype: int64

Label distribution:
label
0    70653
1     5187
Name: count, dtype: int64


,location,split,time_index,flu,swine,stomach,symptoms,virus,bug,strep,...,tests,thinks,ankle,work,hand,complications,children,start,aja,label
0,wyoming,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,wyoming,train,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,wyoming,train,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,wyoming,train,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,wyoming,train,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
